# Index

```markdown
10.1 GPU memory basics ✓
     Allocated vs reserved memory; caching allocator; peak memory

10.2 Qwen training-step profiling ✓
     Weights, activations, gradients, optimizer states

10.3 Sequence length and memory ✓
     Fixed parameter memory vs input-dependent memory

10.4 Gradient checkpointing ✓
     Activation storage vs recomputation

10.5 FlashAttention ✓
     Blockwise attention; online softmax; SDPA backend selection

10.6 Gradient accumulation ✓
     Micro-batch vs effective batch; gradient averaging; update frequency

10.7 Mixed precision and optimizer memory ✓
     FP32/BF16/FP16; autocast; loss scaling

10.8 Profiling and compute optimization
     Compute vs memory-bandwidth bottlenecks
     PyTorch Profiler; fused operations; torch.compile

10.9 Distributed training
     Data parallelism: Distributed Data Parallel (DDP)
     State sharding: Fully Sharded Data Parallel (FSDP)
                    Zero Redundancy Optimizer (ZeRO)
     Tensor / pipeline / sequence / context parallelism
     GPU communication and synchronization costs

10.10 Large-scale training workflows
      Data-loading bottlenecks; sharded checkpoints; resume/recovery
      Throughput, hardware utilization, and scaling efficiency

10.11 Inference and serving systems
      KV-cache management; continuous batching; PagedAttention
      Prefix caching; speculative decoding
      vLLM / SGLang; quantized and multi-GPU inference
      Latency, throughput, and memory benchmarking
```

# Setup

In [2]:
from pprint import pprint
import os, math, time
import pandas as pd
import torch
import torch.optim as optim
from transformers.utils import logging
from transformers import set_seed

# Device.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = device.type == 'cuda' and torch.cuda.is_bf16_supported()

# Suppress warnings.
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

# Seed.
seed = 42
set_seed(seed)

c:\Users\yana\Desktop\ai-summary\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0917 14:57:41.089000 22672 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


# Memory Usage

## Terminology

- Allocated memory: a memory that is actually occupied.
- Reserved memory: a total memory held by `torch`, including unused cached memory.
- Memory size
  - 1 KiB = kibibyte = $2^{10}$ bytes
  - 1 MiB = mebibyte = $2^{20}$ bytes
  - 1 GiB = gibibyte = $2^{30}$ bytes

## Strategy

| Main memory problem                          | Relevant technique             | What it changes                                                                                                          |
| -------------------------------------------- | ------------------------------ | ------------------------------------------------------------------------------------------------------------------------ |
| Too much gradient / optimizer-state memory | **LoRA — Low-Rank Adaptation** | Freeze base weights and train small adapters instead of updating every parameter. ([Hugging Face][1])                    |
| Too many saved activations                   | **Gradient checkpointing**     | Save fewer intermediates; recompute them during backward, trading additional computation for memory. ([Hugging Face][2]) |
| Large attention-score intermediates          | **FlashAttention**             | Compute attention in blocks without storing the full sequence-by-sequence attention matrix. ([arXiv][3])                 |

[1]: https://huggingface.co/docs/peft/main/conceptual_guides/lora "LoRA · Hugging Face"
[2]: https://huggingface.co/docs/transformers/grad_checkpointing "Gradient checkpointing · Hugging Face"
[3]: https://arxiv.org/abs/2205.14135 "[2205.14135] FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness"


## Measure

### Tensor

In [3]:
# Print device name.
device = torch.device('cuda' if torch.cuda.is_available else 'cpu')
device_name = torch.cuda.get_device_name(device)
print(f"Device: {device_name}\n")

# Allocate one tensor on GPU.
size = (4_096, 4_096)
x = torch.empty(
    size,
    dtype=torch.float32,    # 32 bits = 4 bytes.
    device=device,          # gpu.
)

# Measure the memory usage.
mib = 1024 ** 2
memory_allocated = torch.cuda.memory_allocated(device) / mib, 'MiB'
memory_reserved  = torch.cuda.memory_reserved(device) / mib, 'MiB'
print(
    "After allocation\n"
    f"  - Allocated: {memory_allocated}\n"  # 4096 x 4096 x 4 bytes = 64 MiB.
    f"  - Reserved: {memory_reserved}\n"
)

# After delete.
del x

memory_allocated = torch.cuda.memory_allocated(device) / mib, 'MiB'
memory_reserved  = torch.cuda.memory_reserved(device) / mib, 'MiB'
print(
    "After delete\n"
    f"  - Allocated: {memory_allocated}\n"
    f"  - Reserved: {memory_reserved}\n"
)

Device: NVIDIA GeForce RTX 5090

After allocation
  - Allocated: (64.0, 'MiB')
  - Reserved: (64.0, 'MiB')

After delete
  - Allocated: (0.0, 'MiB')
  - Reserved: (64.0, 'MiB')



### MLP

In [4]:
# Simple model.
model = torch.nn.Sequential(
    torch.nn.Linear(1024, 4096, bias=False),
    torch.nn.GELU(),
    torch.nn.Linear(4096, 1024, bias=False),
).to(device=device, dtype=torch.float32)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Sample token: (B, L, d).
x = torch.randn(8, 128, 1024, device=device)

# Log memory.
def report_memory(stage):
    torch.cuda.synchronize(device)      # wait until queued GPU operations are ended.
    allocated = torch.cuda.memory_allocated(device) / mib
    print(f'{stage:22s}: {allocated:8.1f} MiB')

torch.cuda.reset_peak_memory_stats(device)

report_memory('Before forward')

# Simple test loss: mean squared output, not a language-model objective.
loss = model(x).square().mean()
report_memory('After forward')

loss.backward()
report_memory('After backward')

optimizer.step()
report_memory('After first step')

optimizer.zero_grad(set_to_none=True)
report_memory('After zero_grad')

peak = torch.cuda.max_memory_allocated(device) / mib
print(f'Peak allocated        : {peak:8.1f} MiB')

Before forward        :     36.0 MiB
After forward         :    104.0 MiB
After backward        :    132.0 MiB
After first step      :    196.0 MiB
After zero_grad       :    164.0 MiB
Peak allocated        :    228.0 MiB


### LLM

In [5]:
from transformers import AutoModelForCausalLM

# Load a separate model for this experiment.
qwen_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-0.5B',
    dtype=torch.float32,
).to(device)

qwen_model.train()

qwen_optimizer = torch.optim.AdamW(
    qwen_model.parameters(),
    lr=1e-5,
)

# Synthetic token IDs: one sequence of 128 tokens.
B, L = 1, 128

input_ids = torch.randint(
    low=0,
    high=qwen_model.config.vocab_size,
    size=(B, L),
    dtype=torch.long,
    device=device,
)

torch.cuda.reset_peak_memory_stats(device)
report_memory('Before forward')

# Forward and next-token loss.
outputs = qwen_model(
    input_ids=input_ids,
    labels=input_ids,
    use_cache=False,
)

loss = outputs.loss
report_memory('After forward')

# Backward.
loss.backward()
del outputs, loss
report_memory('After backward')

# First optimizer update.
qwen_optimizer.step()
report_memory('After first step')

# Release gradient tensors.
qwen_optimizer.zero_grad(set_to_none=True)
report_memory('After zero_grad')

peak = torch.cuda.max_memory_allocated(device) / mib
print(f'Peak allocated: {peak:.1f} MiB')

Before forward        :   2053.0 MiB
After forward         :   2568.2 MiB
After backward        :   3951.4 MiB
After first step      :   7723.3 MiB
After zero_grad       :   5824.8 MiB
Peak allocated: 9608.6 MiB


### LLM, by component

## vs Sequence Length

| Tensor                                           | Shape          | Effect of L x2  |
| ------------------------------------------------ | -------------- | ------------------------- |
| Hidden states                                    | `(B, L, d)`    | x2      |
| Vocabulary logits                                | `(B, L, V)`    | x2      |
| Full attention-score matrix, **if materialized** | `(B, H, L, L)` | x4 |

> Note) A **materialized** means the intermediate variables are actually **instantiated**.

> Note) Actual allocated memory could be different by various reasons.

In [ ]:
B = 1
qwen_model.train()

for L in [128, 256, 512, 1024]:
    # Clear gradients in the memory.
    qwen_optimizer.zero_grad(set_to_none=True)

    # Random input ids.
    input_ids = torch.randint(
        low=0,
        high=qwen_model.config.vocab_size,
        size=(B, L),
        device=device,
    )

    # Memory before forward pass.
    torch.cuda.synchronize(device)
    baseline = torch.cuda.memory_allocated(device)
    torch.cuda.reset_peak_memory_stats(device)

    # Forward + backward.
    outputs = qwen_model(
        input_ids=input_ids,
        labels=input_ids,
        use_cache=False,
    )
    outputs.loss.backward()

    # Highest allocated memory during this experiment.
    torch.cuda.synchronize(device)
    peak = torch.cuda.max_memory_allocated(device)

    print(
        f'L = {L:4d} | '
        f'Baseline: {(baseline) / mib:8.1f} MiB | '
        f'Peak: {peak / mib:8.1f} MiB'
    )

    # Release this experiment's outputs and inputs.
    del outputs, input_ids

L =  128 | Baseline:   5824.8 MiB | Peak:   8829.2 MiB
L =  256 | Baseline:   5824.8 MiB | Peak:   8916.8 MiB
L =  512 | Baseline:   5824.8 MiB | Peak:   9069.7 MiB
L = 1024 | Baseline:   5824.8 MiB | Peak:  12277.5 MiB


# Gradient Checkpointing

- Without checkpointing
  - Forward: input -> compute activations -> store activations
  - Backward: reuse stored activations
  - More memory, less computation
  - `model.gradient_checkpointing_disable()`
- With checkpointing
  - Forward: input -> compute activations -> store inputs
  - Backward: recompute required activations
  - Less memory, more computation
  - `model.gradient_checkpointing_enable()`

In [12]:
input_ids = torch.randint(
    0,
    qwen_model.config.vocab_size,
    (1, 1024),
    device=device,
)

qwen_model.train()

for enabled in [False, True]:
    qwen_optimizer.zero_grad(set_to_none=True)

    if enabled:
        qwen_model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={'use_reentrant': False},     # use torch's recommended implementation.
        )
    else:
        qwen_model.gradient_checkpointing_disable()

    torch.cuda.synchronize(device)
    baseline = torch.cuda.memory_allocated(device)
    torch.cuda.reset_peak_memory_stats(device)

    outputs = qwen_model(
        input_ids=input_ids,
        labels=input_ids,
        use_cache=False,
    )
    outputs.loss.backward()

    torch.cuda.synchronize(device)
    peak = torch.cuda.max_memory_allocated(device)

    print(
        f'Checkpointing={enabled} | '
        f'Peak: {peak / mib:.1f} MiB | '
        f'Above baseline: {(peak - baseline) / mib:.1f} MiB'
    )

    del outputs
    qwen_optimizer.zero_grad(set_to_none=True)

# Restore the baseline setting.
qwen_model.gradient_checkpointing_disable()
del input_ids

Checkpointing=False | Peak: 12277.5 MiB | Above baseline: 6452.7 MiB
Checkpointing=True | Peak: 9365.9 MiB | Above baseline: 3541.0 MiB


# FlashAttention

- VRAM Memory
  - Main VRAM
  - On-chip SRAM (static RAM): much smaller and faster.
- FlashAttention
  - **Tiles**: divides matrices into subunits, that fit in SRAM.
  - **Online softmax**: incremental computation of the softmax, since it originally requires the whole matrix.
- `sdpa`
  - Scaled Dot-Product Attention.
  - `torch`'s implementation on FlashAttention.

> Note) `qwen_model.set_attn_implementation('sdpa')` does not guarantee using FlashAttention.  
> `torch` automatically chooses based on the inputs and environments.

In [13]:
qwen_model.set_attn_implementation('sdpa')

# Gradient Accumulation

- Calculating a gradient for the **whole batch** requires a large memory.
- Instead calculate and accumulate by small samples -> update weights at once.
  - Micro-batch size = a size of small samples
  - Effective batch size: $B_{effective} = B_{micro} \times \text{accumulation steps}$

In [ ]:
micro_batch_size = 1
accumulation_steps = 4
L = 256

qwen_model.train()
qwen_optimizer.zero_grad(set_to_none=True)

# Iterate over micro-batches.
for _ in range(accumulation_steps):
    # Do NOT call optimizer.zero_grads() for accumulating.

    # One micro-batch.
    input_ids = torch.randint(
        0,
        qwen_model.config.vocab_size,
        (micro_batch_size, L),
        device=device,
    )

    with torch.autocast('cuda', dtype=torch.bfloat16):
        outputs = qwen_model(
            input_ids=input_ids,
            labels=input_ids,
            use_cache=False,
        )

        loss = outputs.loss / accumulation_steps    # average, for one micro-batch loss.

    loss.backward()
    del outputs, loss, input_ids

# Update only after all four micro-batches.
qwen_optimizer.step()
qwen_optimizer.zero_grad(set_to_none=True)

# Mixed Precision

- AMP: Automatic Mixed Precision
  - `with torch.autocast()`
  - Stored weights: FP32
  - Linear/matrix operations: often BF16
  - Sensitive operations (e.g. optimizer): often FP32
- Gradient Scaling
  - `torch.amp.GradScaler()`
  - For `fp16`, when a loss $L$ is too small, the gradient $g$ can be underflowed.
  - Scales the loss and back
    - $L \rightarrow S \times L$
    - Compute $g$
    - $g \leftarrow \frac{g}{S}$

In [ ]:
amp_dtype = torch.bfloat16

scaler = torch.amp.GradScaler(
    'cuda',
    enabled=(amp_dtype == torch.float16),
)

qwen_model.train()
qwen_optimizer.zero_grad(set_to_none=True)

input_ids = torch.randint(
    0,
    qwen_model.config.vocab_size,
    (1, 256),
    device=device,
)

# AMP for forward linear operations.
with torch.autocast('cuda', dtype=amp_dtype):
    outputs = qwen_model(
        input_ids=input_ids,
        labels=input_ids,
        use_cache=False,
    )

# Backward and update stay outside autocast.
scaler.scale(outputs.loss).backward()
scaler.step(qwen_optimizer)
scaler.update()

qwen_optimizer.zero_grad(set_to_none=True)
del outputs, input_ids

# Profiling

- Profiling: What makes it slow?
  - Compute-bound
    - GPU performs arithmetic.
    - Needs faster / fewer calculations.
  - Memory bound
    - Read / write bottleneck for memory I/O.
    - Fewer memory transfers or reuse data already loaded.

## torch.profiler

In [19]:
from torch.profiler import profile, record_function, ProfilerActivity

# Input.
input_ids = torch.randint(
    low=0,
    high=qwen_model.vocab_size,
    size=(1, 512),  # (B, L)
    device=device,
)

# Train.
qwen_model.train()

def training_step():
    qwen_optimizer.zero_grad(set_to_none=True)

    with record_function('forward'):    # 'forward' = custom name, will appear in the result.
        # Forward.
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss = qwen_model(
                input_ids=input_ids,
                labels=input_ids,
                use_cache=False,
            ).loss

        # Backward.
        with record_function('backward'):
            loss.backward()

        # Optimizer.
        with record_function('optimizer'):
            qwen_optimizer.step()

# Warm-up.
warmup_steps = 2
for _ in range(warmup_steps):
    training_step()

torch.cuda.synchronize(device)

# Record one training step.
with profile(
    activities=[
        ProfilerActivity.CPU,
        ProfilerActivity.CUDA,
    ],
    acc_events=True,    # accumulate across multiple recording cycles.
) as prof:
    training_step()
    torch.cuda.synchronize(device)

# Print - GPU time.
print(
    prof.key_averages().table(
        sort_by='self_cuda_time_total',     # accumulated GPU time.
        row_limit=10,
    )
)

# CPU time.
print(
    prof.key_averages().table(
        sort_by='self_cpu_time_total',      # accumulated CPU time.
        row_limit=10,
    )
)

# Delete.
qwen_optimizer.zero_grad(set_to_none=True)
del input_ids

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                forward         0.00%       0.000us         0.00%       0.000us       0.000us     101.866ms       191.75%     101.866ms     101.866ms             1  
                              Optimizer.step#AdamW.step         0.00%       0.000us         0.00%       0.000us       0.000us      27.354ms        51.49%      27.354ms      27.354ms             1  
         